# Front Properties: Global Co-location & Visualization

This notebook co-locates grouped/labeled fronts with mapped physical property
fields (vorticity, Rossby number, SSH, …), then visualises the results.

**Workflow**
1. Load pre-computed labeled fronts + geometric properties (from `process_global_fronts.py`)
2. **[USER]** Point to your mapped property `.npy` files
3. Run `colocate_fronts_with_properties` (with optional dilation)
4. Select a regional view
5. Map – raw property field background + labeled fronts overlay
6. PDFs of per-front property statistics for the selected region
7. Scatter plots across the full global front catalogue

**Prerequisites:** run `process_global_fronts.py` to generate labeled array + parquet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
import xarray as xr
import pandas as pd
from pathlib import Path
import json, re
from scipy.stats import gaussian_kde

from fronts.properties.colocation import colocate_fronts_with_properties

plt.rcParams['figure.figsize'] = (16, 9)
plt.rcParams['font.size'] = 10
%matplotlib inline

print('Imports OK')

## 1. Load Pre-Computed Results

Loads the labeled-fronts array, geometric-property parquet, and lat/lon
coordinates produced by `process_global_fronts.py`.

In [ ]:
# ─── FILE PATHS ─── update these ───────────────────────────────────────────
results_dir  = '/path/to/results/'          # directory with parquet / npy / json
coords_file  = '/path/to/LLC_coords_lat_lon.nc'
# Use a reference file (e.g. divb2) whose name contains the timestamp
ref_file     = '/path/to/LLC4320_2012-11-09T12_00_00_Divb2_v0.nc'
# ─────────────────────────────────────────────────────────────────────────────

# Extract timestamp from ref_file name
pattern = r'(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2})'
match   = re.search(pattern, ref_file)
if not match:
    raise ValueError(f'Cannot extract timestamp from: {ref_file}')
time_str      = match.group(1).replace('_', ':')   # 2012-11-09T12:00:00
time_str_safe = match.group(1).replace('-', '').replace(':', '')  # 20121109T12_00_00
time_str_safe = re.sub(r'T(\d{2})(\d{2})(\d{2})', r'T\1_\2_\3',
                       time_str.replace('-','').replace(':',''), count=1)
print(f'Timestep: {time_str}  (safe: {time_str_safe})')

# Construct standard filenames
metadata_file   = Path(results_dir) / f'metadata_{time_str_safe}.json'
labeled_file    = Path(results_dir) / f'labeled_fronts_global_{time_str_safe}.npy'
properties_file = Path(results_dir) / f'global_front_properties_{time_str_safe}.parquet'

# Load metadata
with open(metadata_file) as f:
    metadata = json.load(f)
downsample_factor = metadata.get('downsample_factor', None)
print(f"Shape: {metadata['shape']},  Fronts: {metadata['num_fronts']:,}")
if downsample_factor:
    print(f'  ⚠️  Downsampled ×{downsample_factor}')

# Load labeled array
labeled_global = np.load(labeled_file)
print(f'labeled_global: {labeled_global.shape}')

# Load geometric properties
df_global = pd.read_parquet(properties_file)
print(f'df_global: {len(df_global):,} fronts × {len(df_global.columns)} cols')
print(f'  Columns: {list(df_global.columns)}')

# Load coordinates
ds_coords  = xr.open_dataset(coords_file)
lat_global = ds_coords['lat'].values if 'lat' in ds_coords else ds_coords['YC'].values
lon_global = ds_coords['lon'].values if 'lon' in ds_coords else ds_coords['XC'].values
ds_coords.close()
if downsample_factor:
    lat_global = lat_global[::downsample_factor, ::downsample_factor]
    lon_global = lon_global[::downsample_factor, ::downsample_factor]

# Roll arrays so columns run -180 → +180
sample_row  = lat_global.shape[0] // 2
min_lon_col = int(np.argmin(lon_global[sample_row, :]))
shift       = -min_lon_col
if min_lon_col != 0:
    print(f'Rolling by {shift} cols to align longitude axis')
    lon_global    = np.roll(lon_global,    shift, axis=1)
    lat_global    = np.roll(lat_global,    shift, axis=1)
    labeled_global = np.roll(labeled_global, shift, axis=1)
else:
    print('No rolling needed')

print(f'\n✓ Pre-computed results loaded.  Array shape: {labeled_global.shape}')

## 2. Load Physical Property Maps

**Edit the dictionary below** to point to your mapped property NetCDF files.
Each entry maps a short property name to a `(filepath, variable_name)` tuple.
The short name becomes the column-name prefix in the co-location output
(e.g. `'relative_vorticity'` → columns `relative_vorticity_mean`, `relative_vorticity_std`, …).

You can also mix in pre-loaded arrays or `.npy` paths — see the commented examples.

**Tip:** to inspect a file's variable names run:
```python
import xarray as xr; print(list(xr.open_dataset('yourfile.nc').data_vars))
```

In [ ]:
# ─── USER: fill in your property file paths & variable names ────────────────
# Each value is a (filepath, variable_name) tuple for a NetCDF (.nc) file.
# The variable_name must match what is inside the NetCDF file.
# To inspect a file's variable names:
#   import xarray as xr; print(list(xr.open_dataset('yourfile.nc').data_vars))

property_files = {
    'relative_vorticity': (
        '/path/to/LLC4320_2012-11-09T12_00_00_relative_vorticity_v1.nc',
        'relative_vorticity',
    ),
    'rossby_number': (
        '/path/to/LLC4320_2012-11-09T12_00_00_rossby_number_v1.nc',
        'rossby_number',
    ),
    # Add more properties here — copy & paste the pattern above:
    # 'SSH': (
    #     '/path/to/LLC4320_2012-11-09T12_00_00_SSH_v1.nc',
    #     'SSH',
    # ),
    # 'strain_rate': (
    #     '/path/to/LLC4320_2012-11-09T12_00_00_strain_rate_v1.nc',
    #     'strain_rate',
    # ),
    # .npy files and pre-loaded arrays also work:
    # 'my_field': '/path/to/my_field.npy',
    # 'preloaded': some_numpy_array,
}
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import xarray as xr
from fronts.properties.colocation import _load_property_file

property_arrays = {}
for prop_name, src in property_files.items():
    arr = _load_property_file(src)          # handles .nc tuples, .npy, or arrays
    arr = arr.squeeze()                      # drop any singleton time/depth dims
    if downsample_factor:
        arr = arr[::downsample_factor, ::downsample_factor]
    if min_lon_col != 0:                     # same longitude roll as labeled array
        arr = np.roll(arr, shift, axis=1)
    property_arrays[prop_name] = arr
    src_label = src[0] if isinstance(src, (list, tuple)) else (
                    str(src) if not isinstance(src, np.ndarray) else '<array>')
    print(f"  {prop_name:25s}  shape={arr.shape}  "
          f"range=[{float(np.nanmin(arr)):.3g}, {float(np.nanmax(arr)):.3g}]")

print(f'\n✓ {len(property_arrays)} property arrays loaded and aligned')

## 3. Co-locate Fronts with Properties

Runs `colocate_fronts_with_properties` over the full global arrays and
merges the result into the existing geometric-property table.

In [ ]:
# ─── CO-LOCATION SETTINGS ────────────────────────────────────────────────────
USE_DILATION    = False   # Dilate each front mask before extracting stats?
DILATION_RADIUS = 3       # Dilation radius in pixels (Euclidean disk);
                          #   only used when USE_DILATION=True
STATS           = ['mean', 'std', 'median']  # any of: mean std median min max count
PERCENTILES     = [10, 90]                   # additional percentile columns
MIN_NPIX        = 5        # skip fronts smaller than this many pixels
NAN_POLICY      = 'omit'   # 'omit' ignores NaN (land/cloud); 'propagate' is faster
# ─────────────────────────────────────────────────────────────────────────────

print('Running co-location…')
df_coloc = colocate_fronts_with_properties(
    labeled_global,
    property_arrays,
    stats=STATS,
    percentiles=PERCENTILES,
    min_npix=MIN_NPIX,
    nan_policy=NAN_POLICY,
    dilation_radius=DILATION_RADIUS if USE_DILATION else 0,
)
print(f'  Co-location complete: {len(df_coloc):,} fronts × {len(df_coloc.columns)} cols')
print(f'  New columns: {[c for c in df_coloc.columns if c not in ("flabel","npix")]}')

# Merge physical stats into geometric properties table
df_enriched = df_global.merge(df_coloc, left_on='label', right_on='flabel', how='inner')
# Resolve duplicate npix columns (geometric parquet uses 'npix'; coloc adds 'npix' too)
if 'npix_x' in df_enriched.columns:
    df_enriched = df_enriched.rename(columns={'npix_x': 'npix'}).drop(columns=['npix_y'], errors='ignore')

print(f'\n✓ Enriched table: {len(df_enriched):,} fronts × {len(df_enriched.columns)} cols')
print(f'  All columns: {list(df_enriched.columns)}')
df_enriched.head(3)

## 4. Select Region for Visualization

Choose **one** of the three options below.

In [ ]:
# ─── VISUALISATION REGION ────────────────────────────────────────────────────
VISUALIZE_ENTIRE_GLOBE = False

USE_LATLON_BOUNDS = False           # Option B: explicit lat/lon bounds
region_bounds = dict(lat_min=-10, lat_max=10, lon_min=-180, lon_max=-160)

USE_PIXEL_REGION = True             # Option C: pixel window around a point
pixel_region = dict(center_lat=-40, center_lon=-30, height=1000, width=1000)
# ─────────────────────────────────────────────────────────────────────────────

if VISUALIZE_ENTIRE_GLOBE:
    i_min, i_max = 0, labeled_global.shape[0]
    j_min, j_max = 0, labeled_global.shape[1]
    region_bounds = dict(lat_min=lat_global.min(), lat_max=lat_global.max(),
                         lon_min=lon_global.min(), lon_max=lon_global.max())

elif USE_LATLON_BOUNDS:
    mask  = ((lat_global >= region_bounds['lat_min']) & (lat_global <= region_bounds['lat_max']) &
             (lon_global >= region_bounds['lon_min']) & (lon_global <= region_bounds['lon_max']))
    r, c  = np.where(mask)
    i_min, i_max = r.min(), r.max() + 1
    j_min, j_max = c.min(), c.max() + 1

elif USE_PIXEL_REGION:
    dist_centre   = np.hypot(lat_global - pixel_region['center_lat'],
                             lon_global - pixel_region['center_lon'])
    ci, cj = np.unravel_index(np.argmin(dist_centre), dist_centre.shape)
    hh, hw = pixel_region['height']//2, pixel_region['width']//2
    i_min = max(0, ci-hh);  i_max = min(labeled_global.shape[0], ci+hh)
    j_min = max(0, cj-hw);  j_max = min(labeled_global.shape[1], cj+hw)
    region_bounds = dict(lat_min=lat_global[i_min:i_max, j_min:j_max].min(),
                         lat_max=lat_global[i_min:i_max, j_min:j_max].max(),
                         lon_min=lon_global[i_min:i_max, j_min:j_max].min(),
                         lon_max=lon_global[i_min:i_max, j_min:j_max].max())

# Regional slices
sl = (slice(i_min, i_max), slice(j_min, j_max))
labeled_viz = labeled_global[sl]
lat_viz     = lat_global[sl]
lon_viz     = lon_global[sl]
prop_viz    = {k: v[sl] for k, v in property_arrays.items()}

# Filter enriched table to region (by centroid)
df_region = df_enriched[
    (df_enriched['centroid_lat'] >= region_bounds['lat_min']) &
    (df_enriched['centroid_lat'] <= region_bounds['lat_max']) &
    (df_enriched['centroid_lon'] >= region_bounds['lon_min']) &
    (df_enriched['centroid_lon'] <= region_bounds['lon_max'])
].copy()

print(f'Region: lat=[{region_bounds["lat_min"]:.2f}, {region_bounds["lat_max"]:.2f}]  '
      f'lon=[{region_bounds["lon_min"]:.2f}, {region_bounds["lon_max"]:.2f}]')
print(f'Array window: {labeled_viz.shape}')
print(f'Fronts in region: {len(df_region):,} / {len(df_enriched):,} global')

## 5. Global Context Map

Shows where the selected region sits on the globe.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 9))
ds = 10   # display downsample factor
extent_g = [lon_global.min(), lon_global.max(), lat_global.min(), lat_global.max()]

# First property as a faint background
bg_key = list(property_arrays.keys())[0]
bg_ds  = property_arrays[bg_key][::ds, ::ds]
vlo, vhi = np.nanpercentile(bg_ds, [5, 95])
ax.imshow(bg_ds, extent=extent_g, origin='lower', aspect='auto',
          cmap='RdBu_r', vmin=vlo, vmax=vhi, alpha=0.4, interpolation='nearest')

# Front pixels in dark grey
ax.imshow(labeled_global[::ds, ::ds] > 0, extent=extent_g, origin='lower',
          aspect='auto', cmap='gray_r', alpha=0.5, interpolation='nearest')

# Red rectangle for selected region
if not VISUALIZE_ENTIRE_GLOBE:
    rect = Rectangle((region_bounds['lon_min'], region_bounds['lat_min']),
                     region_bounds['lon_max'] - region_bounds['lon_min'],
                     region_bounds['lat_max'] - region_bounds['lat_min'],
                     fill=False, edgecolor='red', linewidth=2.5)
    ax.add_patch(rect)
    ax.text(region_bounds['lon_min'], region_bounds['lat_max'] + 3,
            'Selected region', color='red', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.set_title(f'Global fronts – background: {bg_key}', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Regional Map: Property Field + Fronts Overlay

The **background** shows the raw property field as a 2-D image.
Fronts are overlaid, optionally colour-coded by a per-front statistic.

Change `MAP_BG_PROPERTY`, `FRONT_COLOR_BY`, and `FRONT_COLOR_MODE` below.

In [ ]:
# ─── MAP SETTINGS ────────────────────────────────────────────────────────────
MAP_BG_PROPERTY   = 'relative_vorticity'  # raw 2-D field used as background image
MAP_BG_CMAP       = 'RdBu_r'         # colormap for background
MAP_BG_PCT_CLIP   = 97               # symmetric percentile clip for background

FRONT_COLOR_MODE  = 'by_property'    # 'by_property' | 'uniform' | 'random'
FRONT_COLOR_BY    = 'relative_vorticity_mean'  # column in df_region (only for 'by_property')
FRONT_CMAP        = 'plasma'         # colormap for front overlay
FRONT_ALPHA       = 0.85
# ─────────────────────────────────────────────────────────────────────────────

extent_viz = [lon_viz.min(), lon_viz.max(), lat_viz.min(), lat_viz.max()]
fig, ax = plt.subplots(figsize=(16, 10))

# Background: raw property field
bg = prop_viz[MAP_BG_PROPERTY].astype(float)
clip = np.nanpercentile(np.abs(bg), MAP_BG_PCT_CLIP)
im_bg = ax.imshow(bg, extent=extent_viz, origin='lower', aspect='auto',
                  cmap=MAP_BG_CMAP, vmin=-clip, vmax=clip, alpha=0.9)
plt.colorbar(im_bg, ax=ax, label=MAP_BG_PROPERTY, shrink=0.75)

# Front overlay
fronts_rgba = np.zeros((*labeled_viz.shape, 4), dtype=float)

if FRONT_COLOR_MODE == 'uniform':
    fronts_rgba[labeled_viz > 0] = [1.0, 0.3, 0.3, FRONT_ALPHA]

elif FRONT_COLOR_MODE == 'random':
    ulbls = np.unique(labeled_viz[labeled_viz > 0])
    rng   = np.random.default_rng(42)
    clrs  = rng.random((len(ulbls), 3))
    lut_c = np.zeros((int(labeled_viz.max())+1, 3))
    for i, l in enumerate(ulbls): lut_c[l] = clrs[i]
    fronts_rgba[labeled_viz > 0, :3] = lut_c[labeled_viz[labeled_viz > 0]]
    fronts_rgba[labeled_viz > 0,  3] = FRONT_ALPHA

else:  # by_property
    if FRONT_COLOR_BY not in df_region.columns:
        raise ValueError(f'{FRONT_COLOR_BY!r} not in df_region. '
                         f'Available: {list(df_region.columns)}')
    prop_series = df_region.set_index('label')[FRONT_COLOR_BY].dropna()
    max_lbl = int(labeled_viz.max())
    lut_v   = np.full(max_lbl + 1, np.nan)
    for lbl, val in prop_series.items():
        if 0 < lbl <= max_lbl: lut_v[lbl] = val
    pimg  = lut_v[labeled_viz]
    valid = pimg[labeled_viz > 0]; valid = valid[~np.isnan(valid)]
    if len(valid):
        norm = mcolors.Normalize(vmin=np.nanmin(valid), vmax=np.nanmax(valid))
        cmap_f = cm.get_cmap(FRONT_CMAP)
        mask_ok = (labeled_viz > 0) & ~np.isnan(pimg)
        fronts_rgba[mask_ok] = cmap_f(norm(pimg[mask_ok]))
        fronts_rgba[mask_ok, 3] = FRONT_ALPHA
        sm = cm.ScalarMappable(cmap=cmap_f, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, shrink=0.75,
                     label=FRONT_COLOR_BY.replace('_', ' '))

ax.imshow(fronts_rgba, extent=extent_viz, origin='lower', aspect='auto')
dil_tag = f' (dilation={DILATION_RADIUS}px)' if USE_DILATION else ''
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.set_title(f'Background: {MAP_BG_PROPERTY}  |  '
             f'Fronts: {FRONT_COLOR_MODE}{dil_tag}', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--', color='white', linewidth=0.5)
plt.tight_layout(); plt.show()

## 7. PDFs of Per-Front Properties in Selected Region

Each panel shows the distribution of a per-front statistic for fronts
whose centroid falls inside the selected region.
Add or remove entries in `PDF_PROPS` to change what is plotted.

In [ ]:
# ─── PDF SETTINGS ────────────────────────────────────────────────────────────
# Any column in df_region (geometric or co-located physical property stat)
PDF_PROPS = [
    'relative_vorticity_mean',   # mean relative vorticity of each front
    'relative_vorticity_std',    # within-front variability
    'rossby_number_mean',        # mean Rossby number
    'length_km',                 # geometric (from pre-computed parquet)
    'orientation',               # geometric
    # Add any column from df_region, e.g.:
    # 'SSH_mean', 'strain_rate_mean', 'relative_vorticity_p10', ...
]
PDF_NCOLS    = 3
PDF_BINS     = 40
PDF_SHOW_KDE = True     # overlay KDE curve
# ─────────────────────────────────────────────────────────────────────────────

# Remove entries that are not actually present
pdf_avail = [p for p in PDF_PROPS if p in df_region.columns]
if len(pdf_avail) < len(PDF_PROPS):
    missing_p = set(PDF_PROPS) - set(pdf_avail)
    print(f'⚠️  Skipping unavailable columns: {missing_p}')

nrows = int(np.ceil(len(pdf_avail) / PDF_NCOLS))
fig, axes = plt.subplots(nrows, PDF_NCOLS, figsize=(5*PDF_NCOLS, 4*nrows))
axes_flat  = np.array(axes).flatten()

for idx, prop in enumerate(pdf_avail):
    ax   = axes_flat[idx]
    data = df_region[prop].dropna()
    ax.hist(data, bins=PDF_BINS, density=True, alpha=0.65,
            color='steelblue', edgecolor='white', linewidth=0.4)
    if PDF_SHOW_KDE and len(data) > 20:
        try:
            kde = gaussian_kde(data)
            xs  = np.linspace(data.min(), data.max(), 300)
            ax.plot(xs, kde(xs), 'r-', lw=2, label='KDE')
        except Exception:
            pass
    ax.axvline(data.mean(),   color='navy', ls='--', lw=1.5, label=f'mean={data.mean():.3g}')
    ax.axvline(data.median(), color='darkorange', ls=':', lw=1.5,
               label=f'med={data.median():.3g}')
    ax.set_title(f'{prop}  (n={len(data):,})', fontsize=9)
    ax.set_xlabel(prop.replace('_', ' ')); ax.set_ylabel('Density')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

for idx in range(len(pdf_avail), len(axes_flat)):
    axes_flat[idx].set_visible(False)

dil_tag = f' (dilation={DILATION_RADIUS}px)' if USE_DILATION else ''
fig.suptitle(f'Per-front property PDFs – selected region{dil_tag}  '
             f'(n={len(df_region):,} fronts)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 8. Scatter Plots – Global Front Catalogue

These plots use **all globally detected fronts** (not just the region).
Edit `SCATTER_PAIRS` to explore different variable combinations.
Each tuple is `(x_column, y_column, color_column_or_None)`.

In [ ]:
# ─── SCATTER SETTINGS ─────────────────────────────────────────────────────────
# Each row: (x_column, y_column, color_by_or_None)
# x/y can be ANY column in df_enriched (geometric props + co-location stats)
SCATTER_PAIRS = [
    ('centroid_lat',  'relative_vorticity_mean', None),
    ('length_km',     'rossby_number_mean',      None),
    ('centroid_lat',  'rossby_number_mean',      'length_km'),
    ('length_km',     'relative_vorticity_mean', None),
    # More examples — uncomment or add your own:
    # ('centroid_lat', 'SSH_mean',                    None),
    # ('orientation',  'relative_vorticity_std',      'rossby_number_mean'),
    # ('length_km',    'relative_vorticity_p90',      None),
]

SCATTER_ALPHA   = 0.15      # point transparency (lower = better for dense clouds)
SCATTER_SIZE    = 1.5       # marker size
SCATTER_CMAP    = 'viridis' # colormap used when color_by is not None
N_SAMPLE        = 20_000    # subsample for speed; set None to use all fronts
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Subsample for fast rendering
if N_SAMPLE and len(df_enriched) > N_SAMPLE:
    df_plot = df_enriched.sample(N_SAMPLE, random_state=42)
    print(f'Plotting {N_SAMPLE:,} / {len(df_enriched):,} fronts (random subsample)')
else:
    df_plot = df_enriched
    print(f'Plotting all {len(df_enriched):,} fronts')

# Filter to pairs whose columns actually exist
valid_pairs = []
for (xc, yc, cc) in SCATTER_PAIRS:
    missing_cols = [c for c in [xc, yc] + ([cc] if cc else [])
                    if c not in df_plot.columns]
    if missing_cols:
        print(f'  ⚠️  Skipping ({xc}, {yc}): missing columns {missing_cols}')
    else:
        valid_pairs.append((xc, yc, cc))

ncols = min(2, len(valid_pairs))
nrows = int(np.ceil(len(valid_pairs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(8*ncols, 5*nrows))
axes_flat  = np.array(axes).flatten() if len(valid_pairs) > 1 else [axes]

for idx, (xc, yc, cc) in enumerate(valid_pairs):
    ax   = axes_flat[idx]
    sub  = df_plot[[xc, yc] + ([cc] if cc else [])].dropna()
    if cc:
        sc = ax.scatter(sub[xc], sub[yc], c=sub[cc],
                        cmap=SCATTER_CMAP, s=SCATTER_SIZE, alpha=SCATTER_ALPHA,
                        rasterized=True)
        plt.colorbar(sc, ax=ax, label=cc.replace('_', ' '), shrink=0.8)
    else:
        ax.scatter(sub[xc], sub[yc], s=SCATTER_SIZE, alpha=SCATTER_ALPHA,
                   color='steelblue', rasterized=True)
    ax.set_xlabel(xc.replace('_', ' ')); ax.set_ylabel(yc.replace('_', ' '))
    ax.set_title(f'{yc}  vs  {xc}  (n={len(sub):,})', fontsize=9)
    ax.grid(True, alpha=0.3)

for idx in range(len(valid_pairs), len(axes_flat)):
    axes_flat[idx].set_visible(False)

dil_tag = f' (dilation={DILATION_RADIUS}px)' if USE_DILATION else ''
fig.suptitle(f'Global scatter plots{dil_tag}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Summary Statistics

In [ ]:
print('=' * 65)
print('GLOBAL CATALOGUE')
print('=' * 65)
print(f'  Total fronts: {len(df_enriched):,}')
phys_cols = [c for c in df_enriched.columns
             if any(c.endswith(f'_{s}') for s in ['mean','std','median'])]
if phys_cols:
    print(f'\nPer-front physical statistics (global median ± std):')
    for col in phys_cols:
        d = df_enriched[col].dropna()
        print(f'  {col:30s}  median={d.median():.3g}  std={d.std():.3g}  n={len(d):,}')

print()
print('=' * 65)
print('SELECTED REGION')
print('=' * 65)
print(f'  lat=[{region_bounds["lat_min"]:.2f}, {region_bounds["lat_max"]:.2f}]  '
      f'lon=[{region_bounds["lon_min"]:.2f}, {region_bounds["lon_max"]:.2f}]')
print(f'  Fronts: {len(df_region):,}  ({100*len(df_region)/len(df_enriched):.2f}% of global)')
if phys_cols:
    print(f'\nPer-front physical statistics (region median ± std):')
    for col in phys_cols:
        d = df_region[col].dropna()
        if len(d):
            print(f'  {col:30s}  median={d.median():.3g}  std={d.std():.3g}  n={len(d):,}')
print()